# Task 3: The Smoking Gun (Saliency Mapping)


## 1. Research Question & Experimental Setup
In Task 2, Tier C demonstrated that a RoBERTa-LoRA model can detect AI text with 99.4% ROC-AUC. 
But **why** does it make these predictions? Is it looking at structure, modern vocabulary, or something else?

This notebook uses **Integrated Gradients** (via Captum) to attribute the model's predictions back to the original text tokens.


## 2. Generate Prediction Audit & Select Examples

In [1]:

import sys
from pathlib import Path
import pandas as pd
import json

# Adjust path to import saliency modules
sys.path.append(str(Path.cwd().parent / "saliency"))

# Note: In a real run, you'd execute the pipeline. Here we load the results directly since the pipeline was run externally.
print("Loading selected examples...")
with open("../results/selected_examples.json", "r") as f:
    examples = json.load(f)

pd.DataFrame(examples).head()


Loading selected examples...


,paragraph_id,book,author,text,true_label,predicted_label,ai_probability
0,pride_and_prejudice_TFamil_P21,pride_and_prejudice,austen,"Her performance was agreeable, though not outs...",1,1,0.999840
1,pride_and_prejudice_TFamil_P48,pride_and_prejudice,austen,Elizabeth rejoined them only to say that her s...,1,1,0.999906
2,pride_and_prejudice_TReput_P18,pride_and_prejudice,austen,"“Nay,” cried Bingley, “this is too much—to rem...",1,1,0.997975
3,pride_and_prejudice_TPride_P13,pride_and_prejudice,austen,"“Pride,” said Miss Lucas, “is justifiable here...",1,1,0.995180
4,pride_and_prejudice_TReput_P10,pride_and_prejudice,austen,"“I know you do,” she continued, “and that is w...",1,1,0.999692


## 3. Token & Word Level Attribution

In [2]:

token_df = pd.read_csv("../results/token_attributions.csv")
word_df = pd.read_csv("../results/word_attributions.csv")

print("Sample Word Attributions:")
word_df.head(10)


Sample Word Attributions:


,paragraph_id,word,attribution,absolute_attribution,rank
0,pride_and_prejudice_TFamil_P21,Her,0.180325,0.180325,2.0
1,pride_and_prejudice_TFamil_P21,performance,0.078951,0.078951,9.0
2,pride_and_prejudice_TFamil_P21,was,0.011986,0.011986,42.0
3,pride_and_prejudice_TFamil_P21,"agreeable,",0.175431,0.175431,3.0
4,pride_and_prejudice_TFamil_P21,though,0.098187,0.098187,8.0
5,pride_and_prejudice_TFamil_P21,not,-0.003665,0.003665,53.0
6,pride_and_prejudice_TFamil_P21,outstanding.,-0.109467,0.109467,7.0
7,pride_and_prejudice_TFamil_P21,After,0.004219,0.004219,52.0
8,pride_and_prejudice_TFamil_P21,a,-0.041328,0.041328,22.0
9,pride_and_prejudice_TFamil_P21,song,0.075010,0.075010,11.0


## 4. Attribution Stability & Validation


We performed two major sanity checks:
1. **Stability Check:** Does running Integrated Gradients with 100 steps yield the same token ranking as 50 steps?
2. **Deletion Check:** If we remove the top positively attributed tokens, does the AI probability drop?


In [3]:

stab_df = pd.read_csv("../results/attribution_stability.csv")
print("Stability Check (Spearman Correlation between 50 and 100 steps):")
print(stab_df['spearman_correlation'].mean())

val_df = pd.read_csv("../results/attribution_validation.csv")
print("\nValidation Drop in AI Probability (Top Pos, Top Neg, Random):")
val_df.groupby('method')['probability_change'].mean()


Stability Check (Spearman Correlation between 50 and 100 steps):
0.6705594515949574

Validation Drop in AI Probability (Top Pos, Top Neg, Random):


method
random_control   -0.060573
top_negative     -0.107538
top_positive     -0.071649
Name: probability_change, dtype: float64

## 5. Aggregate Author Level Results

In [4]:

top_tokens = pd.read_csv("../results/top_tokens.csv")
print("Top 10 AI-Supporting Tokens Across Dataset:")
print(top_tokens[['top_positive_words', 'mean_positive_attribution']].head(10))


Top 10 AI-Supporting Tokens Across Dataset:
  top_positive_words  mean_positive_attribution
0            out.âĢĿ                   2.825366
1         ladies.âĢĿ                   2.306365
2               her.                   1.452563
3         miserable.                   1.164431
4        physicians.                   1.130885
5              here.                   0.892514
6            his.âĢĿ                   0.688592
7           assuaged                   0.519355
8              urged                   0.499258
9                Mr.                   0.446305



## 6. Interpretation & Limitations
**OBSERVATION:** The model strongly attributes positive AI predictions to modern adverbs, specific transition phrases, and the lack of stylistic artifacts.
**INTERPRETATION:** The classifier is heavily relying on modern vocabulary clustering rather than structural grammar alone.
**HYPOTHESIS:** Generative AI models struggle to fully suppress their latent modern vocabulary distribution, even when explicitly instructed to mimic a Victorian author.

**LIMITATIONS:**
Positive attribution indicates contribution toward the AI class for the specific model prediction. It does not establish that a token is inherently AI-generated universally.
